# Task D — Attention Shifting & Inhibition of Return

**Track:** Attention — Attention Shifting
**Benchmark:** CogAttention v1.0
**Subtasks:** `shifting` (Rule-Switch Classification), `inhibition_return` (IOR Penalty)

---

## What This Notebook Does

This notebook benchmarks an LLM's **attention shifting** — its ability to flexibly switch between classification rules mid-sequence (detecting perseveration errors) and whether it exhibits inhibition of return (degraded performance when forced to re-attend a previously processed passage).

### Subtask Breakdown

| Subtask | Paradigm | What It Measures |
|---------|----------|-----------------|
| **Rule-Switch Classification** | Wisconsin Card Sorting Test (WCST) / Task Interference (EMNLP 2024) | Presents a sequence of items where the classification rule changes without explicit warning. Model must detect the shift and apply the new rule. Measures perseveration (continuing to apply the old rule after a switch). |
| **Inhibition of Return** | IOR Paradigm (Posner & Cohen, 1984) | Presents passages, then re-queries about a previously attended passage after an intervening task. Measures whether returning to old content incurs an accuracy penalty (IOR effect). |

### Cognitive Science Grounding

- **Task switching / set shifting** (Monsell, 2003; Jersild, 1927): Switching between task rules incurs a "switch cost" — slower and more error-prone responses. WCST perseveration is a hallmark of executive dysfunction.
- **Inhibition of return** (Posner & Cohen, 1984; Klein, 2000): After attending to a location/stimulus, returning to it is slower — an evolutionary mechanism to promote exploration. We test whether LLMs show analogous penalties.
- **Perseveration** (Milner, 1963): Continuing to apply an outdated rule despite feedback — a key measure of cognitive flexibility.

### Difficulty Scaling

| Level | Rule-Switch | Inhibition of Return |
|-------|------------|---------------------|
| Easy | 2 rules, explicit switch cue | Short intervening task, obvious re-query |
| Medium | 3 rules, subtle cue | Moderate intervening content |
| Hard | 4 rules, no explicit cue | Long intervening content, indirect re-query |
| Expert | 5 rules, rapid switching | Multiple passages, nested returns |
| Frontier | 6+ rules, adversarial near-switch patterns | Max-length context, deeply buried return targets |

### Scoring

SDK assertion pass rate = per-element accuracy. Each classification decision and return-query answer is a separate assertion. Perseveration errors are explicitly tracked.

---

`<!-- COGATTENTION-BENCH-CANARY-C834A0326830 -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Attention Shifting
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re

def extract_answer_block(response):
    for pat in [r"ANSWER:\s*(.*)", r"Answer:\s*(.*)", r"answer:\s*(.*)"]:
        match = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return response.strip()

def extract_numbered_answers(response):
    answer_block = extract_answer_block(response)
    results = {}
    matches = re.findall(
        r"(\d+)\s*[.):\-]\s*(.+?)(?=\n\d+\s*[.):\-]|\Z)",
        answer_block, re.DOTALL,
    )
    for num, val in matches:
        results[num] = val.strip().rstrip(".")
    return results

def extract_list_items(response):
    answer_block = extract_answer_block(response)
    bullets = re.findall(r"[-\u2022]\s*(.+?)(?:\n|$)", answer_block)
    if bullets:
        return [b.strip().rstrip(".") for b in bullets]
    numeric_items = re.findall(
        r'[\$]?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*(?:\xb0[CF]|mg/L|%|\$))?',
        answer_block,
    )
    if numeric_items and len(numeric_items) >= 2:
        return [x.strip() for x in numeric_items]
    if "," in answer_block:
        items = [x.strip().rstrip(".") for x in answer_block.split(",")]
        return [x for x in items if x]
    lines = [l.strip().rstrip(".") for l in answer_block.split("\n") if l.strip()]
    return lines if lines else ([answer_block] if answer_block else [])

def extract_person_item_pairs(response):
    answer_block = extract_answer_block(response)
    results = {}
    for pat in [
        r"[-\u2022]?\s*(\w+)\s*:\s*(.+?)(?:\n|$)",
        r"[-\u2022]?\s*(\w+)\s+holds?\s+(?:a\s+)?(.+?)(?:\n|$)",
    ]:
        matches = re.findall(pat, answer_block, re.IGNORECASE)
        if matches:
            for name, item in matches:
                results[name.strip()] = item.strip().rstrip(".")
            break
    return results

def fuzzy_value_match(predicted, gold):
    pred_clean = re.sub(r"\s+", " ", predicted.strip().lower())
    gold_clean = re.sub(r"\s+", " ", gold.strip().lower())
    if pred_clean == gold_clean:
        return True
    if gold_clean in pred_clean:
        return True
    try:
        pred_num = float(re.sub(r"[,$%\xb0]", "", predicted))
        gold_num = float(re.sub(r"[,$%\xb0]", "", gold))
        return pred_num == gold_num
    except (ValueError, TypeError):
        pass
    return False

def _escape_for_regex(s):
    return re.escape(s).replace(r"\ ", r"\s+")


def run_assertions_shifting(response, gold, kbench):
    for idx_str, gold_val in gold["answers"].items():
        pattern = rf"(?i){re.escape(idx_str)}\s*[.):\-]\s*.*{_escape_for_regex(gold_val)}"
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"Item {idx_str} should be classified as '{gold_val}'"
        )


def run_assertions_inhibition_return(response, gold, kbench):
    for idx_str, gold_val in gold["answers"].items():
        pattern = rf"(?i){re.escape(idx_str)}\s*[.):\-]\s*.*{_escape_for_regex(gold_val)}"
        phase = "unknown"
        for q in gold["questions"]:
            if str(q["question_num"]) == idx_str:
                phase = q["phase"]
                break
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"Q{idx_str} ({phase}) should be '{gold_val}'"
        )


print("CogAttention helpers loaded")
print(f"Task types: ['shifting', 'inhibition_return']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_shifting")
def cogattention_shifting(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention shifting task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_shifting(response, gold, kbench)


@kbench.task(name="cogattention_inhibition_return")
def cogattention_inhibition_return(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention inhibition_return task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_inhibition_return(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "shifting_easy_000",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. candle → ?\n2. mirror → ?\n3. shovel → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n4. rabbit → ?\n5. eagle → ?\n6. wrench → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"object\", \"2\": \"object\", \"3\": \"object\", \"4\": \"late\", \"5\": \"early\", \"6\": \"late\"}}"
 },
 {
  "task_id": "shifting_easy_001",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. mango → ?\n2. banana → ?\n3. basket → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by CATEGORY: animal, food, or object\n\n4. turnip → ?\n5. tiger → ?\n6. orange → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"early\", \"3\": \"early\", \"4\": \"food\", \"5\": \"animal\", \"6\": \"food\"}}"
 },
 {
  "task_id": "shifting_easy_002",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. rabbit → ?\n2. lobster → ?\n3. shovel → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by CATEGORY: animal, food, or object\n\n4. wrench → ?\n5. pepper → ?\n6. candle → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"late\", \"2\": \"early\", \"3\": \"late\", \"4\": \"object\", \"5\": \"food\", \"6\": \"object\"}}"
 },
 {
  "task_id": "shifting_easy_003",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. needle → ?\n2. shovel → ?\n3. walnut → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n4. hammer → ?\n5. orange → ?\n6. mirror → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"object\", \"2\": \"object\", \"3\": \"food\", \"4\": \"early\", \"5\": \"late\", \"6\": \"early\"}}"
 },
 {
  "task_id": "shifting_easy_004",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. needle → ?\n2. tiger → ?\n3. falcon → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n4. mirror → ?\n5. orange → ?\n6. lobster → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"object\", \"2\": \"animal\", \"3\": \"animal\", \"4\": \"early\", \"5\": \"late\", \"6\": \"early\"}}"
 },
 {
  "task_id": "shifting_easy_005",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. lobster → ?\n2. candle → ?\n3. shovel → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n4. hammer → ?\n5. wrench → ?\n6. banana → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"object\", \"3\": \"object\", \"4\": \"early\", \"5\": \"late\", \"6\": \"early\"}}"
 },
 {
  "task_id": "shifting_easy_006",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. parrot → ?\n2. wrench → ?\n3. cherry → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n4. needle → ?\n5. turnip → ?\n6. rabbit → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"object\", \"3\": \"food\", \"4\": \"late\", \"5\": \"late\", \"6\": \"late\"}}"
 },
 {
  "task_id": "shifting_easy_007",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. hammer → ?\n2. eagle → ?\n3. mirror → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by CATEGORY: animal, food, or object\n\n4. shovel → ?\n5. falcon → ?\n6. walnut → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"early\", \"3\": \"early\", \"4\": \"object\", \"5\": \"animal\", \"6\": \"food\"}}"
 },
 {
  "task_id": "shifting_medium_008",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. cherry → ?\n2. needle → ?\n3. falcon → ?\n4. eagle → ?\n\nRule change: Classify each word by CATEGORY: animal, food, or object\n\n5. donkey → ?\n6. pigeon → ?\n7. orange → ?\n8. banana → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"late\", \"3\": \"early\", \"4\": \"early\", \"5\": \"animal\", \"6\": \"animal\", \"7\": \"food\", \"8\": \"food\"}}"
 },
 {
  "task_id": "shifting_medium_009",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. walnut → ?\n2. rabbit → ?\n3. pepper → ?\n4. needle → ?\n\nRule change: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n5. pigeon → ?\n6. candle → ?\n7. salmon → ?\n8. orange → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"animal\", \"3\": \"food\", \"4\": \"object\", \"5\": \"late\", \"6\": \"early\", \"7\": \"late\", \"8\": \"late\"}}"
 },
 {
  "task_id": "shifting_medium_010",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. walnut → ?\n2. mirror → ?\n3. needle → ?\n4. shovel → ?\n\nRule change: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n5. basket → ?\n6. rabbit → ?\n7. turnip → ?\n8. mango → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"object\", \"3\": \"object\", \"4\": \"object\", \"5\": \"early\", \"6\": \"late\", \"7\": \"late\", \"8\": \"early\"}}"
 },
 {
  "task_id": "shifting_medium_011",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. eagle → ?\n2. wrench → ?\n3. turnip → ?\n4. basket → ?\n\nRule change: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n5. candle → ?\n6. mango → ?\n7. pigeon → ?\n8. cherry → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"object\", \"3\": \"food\", \"4\": \"object\", \"5\": \"early\", \"6\": \"early\", \"7\": \"late\", \"8\": \"early\"}}"
 },
 {
  "task_id": "shifting_medium_012",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. pigeon → ?\n2. mirror → ?\n3. lobster → ?\n4. pepper → ?\n\nRule change: Classify each word by CATEGORY: animal, food, or object\n\n5. pencil → ?\n6. needle → ?\n7. cherry → ?\n8. basket → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"late\", \"2\": \"early\", \"3\": \"early\", \"4\": \"late\", \"5\": \"object\", \"6\": \"object\", \"7\": \"food\", \"8\": \"object\"}}"
 },
 {
  "task_id": "shifting_medium_013",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. candle → ?\n2. parrot → ?\n3. falcon → ?\n4. mango → ?\n\nRule change: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n5. shovel → ?\n6. walnut → ?\n7. donkey → ?\n8. wrench → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"object\", \"2\": \"animal\", \"3\": \"animal\", \"4\": \"food\", \"5\": \"late\", \"6\": \"late\", \"7\": \"early\", \"8\": \"late\"}}"
 },
 {
  "task_id": "shifting_medium_014",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. tiger → ?\n2. basket → ?\n3. parrot → ?\n4. lobster → ?\n\nRule change: Classify each word by CATEGORY: animal, food, or object\n\n5. walnut → ?\n6. pigeon → ?\n7. salmon → ?\n8. pepper → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"late\", \"2\": \"early\", \"3\": \"late\", \"4\": \"early\", \"5\": \"food\", \"6\": \"animal\", \"7\": \"food\", \"8\": \"food\"}}"
 },
 {
  "task_id": "shifting_medium_015",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. mango → ?\n2. cherry → ?\n3. tiger → ?\n4. walnut → ?\n\nRule change: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n5. falcon → ?\n6. turnip → ?\n7. pepper → ?\n8. pencil → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"food\", \"3\": \"animal\", \"4\": \"food\", \"5\": \"early\", \"6\": \"late\", \"7\": \"late\", \"8\": \"late\"}}"
 },
 {
  "task_id": "shifting_hard_016",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. rabbit → ?\n2. basket → ?\n3. orange → ?\n4. wrench → ?\n5. shovel → ?\n\n(Note: from this point, classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more)\n\n6. pigeon → ?\n7. mango → ?\n8. needle → ?\n9. lobster → ?\n10. salmon → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"object\", \"3\": \"food\", \"4\": \"object\", \"5\": \"object\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_hard_017",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. pepper → ?\n2. eagle → ?\n3. wrench → ?\n4. banana → ?\n5. rabbit → ?\n\n(Note: from this point, classify each word by category: animal, food, or object)\n\n6. cherry → ?\n7. shovel → ?\n8. hammer → ?\n9. pigeon → ?\n10. mirror → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"1\", \"4\": \"3+\", \"5\": \"2\", \"6\": \"food\", \"7\": \"object\", \"8\": \"object\", \"9\": \"animal\", \"10\": \"object\"}}"
 },
 {
  "task_id": "shifting_hard_018",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. parrot → ?\n2. turnip → ?\n3. pigeon → ?\n4. banana → ?\n5. rabbit → ?\n\n(Note: from this point, classify each word by category: animal, food, or object)\n\n6. candle → ?\n7. pepper → ?\n8. lobster → ?\n9. orange → ?\n10. mirror → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"2\", \"4\": \"3+\", \"5\": \"2\", \"6\": \"object\", \"7\": \"food\", \"8\": \"animal\", \"9\": \"food\", \"10\": \"object\"}}"
 },
 {
  "task_id": "shifting_hard_019",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. mango → ?\n2. rabbit → ?\n3. turnip → ?\n4. tiger → ?\n5. needle → ?\n\n(Note: from this point, classify each word by category: animal, food, or object)\n\n6. basket → ?\n7. pencil → ?\n8. eagle → ?\n9. banana → ?\n10. pepper → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"2\", \"4\": \"2\", \"5\": \"2\", \"6\": \"object\", \"7\": \"object\", \"8\": \"animal\", \"9\": \"food\", \"10\": \"food\"}}"
 },
 {
  "task_id": "shifting_hard_020",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. lobster → ?\n2. shovel → ?\n3. tiger → ?\n4. hammer → ?\n5. basket → ?\n\n(Note: from this point, classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more)\n\n6. walnut → ?\n7. mirror → ?\n8. pigeon → ?\n9. salmon → ?\n10. candle → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"object\", \"3\": \"animal\", \"4\": \"object\", \"5\": \"object\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_hard_021",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. candle → ?\n2. falcon → ?\n3. shovel → ?\n4. mango → ?\n5. pepper → ?\n\n(Note: from this point, classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more)\n\n6. eagle → ?\n7. basket → ?\n8. mirror → ?\n9. parrot → ?\n10. tiger → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"object\", \"2\": \"animal\", \"3\": \"object\", \"4\": \"food\", \"5\": \"food\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_hard_022",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. donkey → ?\n2. pepper → ?\n3. lobster → ?\n4. hammer → ?\n5. orange → ?\n\n(Note: from this point, classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more)\n\n6. falcon → ?\n7. parrot → ?\n8. cherry → ?\n9. mirror → ?\n10. banana → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"food\", \"3\": \"animal\", \"4\": \"object\", \"5\": \"food\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"3+\"}}"
 },
 {
  "task_id": "shifting_hard_023",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. eagle → ?\n2. lobster → ?\n3. walnut → ?\n4. pigeon → ?\n5. donkey → ?\n\n(Note: from this point, classify each word by category: animal, food, or object)\n\n6. orange → ?\n7. parrot → ?\n8. tiger → ?\n9. banana → ?\n10. pencil → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"2\", \"4\": \"2\", \"5\": \"2\", \"6\": \"food\", \"7\": \"animal\", \"8\": \"animal\", \"9\": \"food\", \"10\": \"object\"}}"
 },
 {
  "task_id": "shifting_expert_024",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. wrench → ?\n2. pencil → ?\n3. parrot → ?\n4. shovel → ?\n5. needle → ?\n\nContinue with the following adjustment — classify each word by first letter: 'early' if a-m, 'late' if n-z\n\n6. salmon → ?\n7. mirror → ?\n8. tiger → ?\n9. walnut → ?\n10. banana → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"1\", \"2\": \"2\", \"3\": \"2\", \"4\": \"2\", \"5\": \"2\", \"6\": \"late\", \"7\": \"early\", \"8\": \"late\", \"9\": \"late\", \"10\": \"early\"}}"
 },
 {
  "task_id": "shifting_expert_025",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. mirror → ?\n2. needle → ?\n3. walnut → ?\n4. tiger → ?\n5. donkey → ?\n\nContinue with the following adjustment — classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. eagle → ?\n7. turnip → ?\n8. wrench → ?\n9. basket → ?\n10. mango → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"late\", \"3\": \"late\", \"4\": \"late\", \"5\": \"early\", \"6\": \"2\", \"7\": \"2\", \"8\": \"1\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_expert_026",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. pepper → ?\n2. pencil → ?\n3. salmon → ?\n4. orange → ?\n5. mirror → ?\n\nContinue with the following adjustment — classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. basket → ?\n7. hammer → ?\n8. tiger → ?\n9. shovel → ?\n10. eagle → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"late\", \"2\": \"late\", \"3\": \"late\", \"4\": \"late\", \"5\": \"early\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_expert_027",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. walnut → ?\n2. shovel → ?\n3. wrench → ?\n4. turnip → ?\n5. orange → ?\n\nContinue with the following adjustment — classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. needle → ?\n7. candle → ?\n8. pencil → ?\n9. donkey → ?\n10. lobster → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"late\", \"2\": \"late\", \"3\": \"late\", \"4\": \"late\", \"5\": \"late\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_expert_028",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. donkey → ?\n2. pepper → ?\n3. parrot → ?\n4. orange → ?\n5. cherry → ?\n\nContinue with the following adjustment — classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. eagle → ?\n7. wrench → ?\n8. tiger → ?\n9. salmon → ?\n10. pigeon → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"late\", \"3\": \"late\", \"4\": \"late\", \"5\": \"early\", \"6\": \"2\", \"7\": \"1\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_expert_029",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. pigeon → ?\n2. lobster → ?\n3. tiger → ?\n4. needle → ?\n5. donkey → ?\n\nContinue with the following adjustment — classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. shovel → ?\n7. walnut → ?\n8. candle → ?\n9. turnip → ?\n10. pepper → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"late\", \"2\": \"early\", \"3\": \"late\", \"4\": \"late\", \"5\": \"early\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_expert_030",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. falcon → ?\n2. turnip → ?\n3. pigeon → ?\n4. wrench → ?\n5. mirror → ?\n\nContinue with the following adjustment — classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. parrot → ?\n7. donkey → ?\n8. orange → ?\n9. banana → ?\n10. rabbit → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"late\", \"3\": \"late\", \"4\": \"late\", \"5\": \"early\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"3+\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_expert_031",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. shovel → ?\n2. lobster → ?\n3. pepper → ?\n4. wrench → ?\n5. needle → ?\n\nContinue with the following adjustment — classify each word by first letter: 'early' if a-m, 'late' if n-z\n\n6. orange → ?\n7. pencil → ?\n8. donkey → ?\n9. banana → ?\n10. falcon → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"2\", \"4\": \"1\", \"5\": \"2\", \"6\": \"late\", \"7\": \"late\", \"8\": \"early\", \"9\": \"early\", \"10\": \"early\"}}"
 },
 {
  "task_id": "shifting_frontier_032",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. orange → ?\n2. pencil → ?\n3. walnut → ?\n4. rabbit → ?\n5. tiger → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n6. candle → ?\n7. lobster → ?\n8. turnip → ?\n9. falcon → ?\n10. salmon → ?\n11. mango → ?\n12. needle → ?\n13. parrot → ?\n14. cherry → ?\n15. hammer → ?\n\nClassify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n16. eagle → ?\n17. pepper → ?\n18. mirror → ?\n19. pigeon → ?\n20. basket → ?\n21. donkey → ?\n22. banana → ?\n23. wrench → ?\n24. shovel → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"object\", \"3\": \"food\", \"4\": \"animal\", \"5\": \"animal\", \"6\": \"early\", \"7\": \"early\", \"8\": \"late\", \"9\": \"early\", \"10\": \"late\", \"11\": \"early\", \"12\": \"late\", \"13\": \"late\", \"14\": \"early\", \"15\": \"early\", \"16\": \"2\", \"17\": \"2\", \"18\": \"2\", \"19\": \"2\", \"20\": \"2\", \"21\": \"2\", \"22\": \"3+\", \"23\": \"1\", \"24\": \"2\"}}"
 },
 {
  "task_id": "shifting_frontier_033",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. basket → ?\n2. tiger → ?\n3. pencil → ?\n4. falcon → ?\n5. hammer → ?\n\nClassify each word by CATEGORY: animal, food, or object\n\n6. mirror → ?\n7. shovel → ?\n8. mango → ?\n9. lobster → ?\n10. pepper → ?\n11. salmon → ?\n12. eagle → ?\n13. candle → ?\n14. walnut → ?\n15. rabbit → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n16. cherry → ?\n17. wrench → ?\n18. banana → ?\n19. turnip → ?\n20. needle → ?\n21. donkey → ?\n22. parrot → ?\n23. pigeon → ?\n24. orange → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"2\", \"4\": \"2\", \"5\": \"2\", \"6\": \"object\", \"7\": \"object\", \"8\": \"food\", \"9\": \"animal\", \"10\": \"food\", \"11\": \"food\", \"12\": \"animal\", \"13\": \"object\", \"14\": \"food\", \"15\": \"animal\", \"16\": \"early\", \"17\": \"late\", \"18\": \"early\", \"19\": \"late\", \"20\": \"late\", \"21\": \"early\", \"22\": \"late\", \"23\": \"late\", \"24\": \"late\"}}"
 },
 {
  "task_id": "shifting_frontier_034",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. falcon → ?\n2. banana → ?\n3. basket → ?\n4. tiger → ?\n5. wrench → ?\n\nClassify each word by CATEGORY: animal, food, or object\n\n6. candle → ?\n7. pigeon → ?\n8. turnip → ?\n9. donkey → ?\n10. shovel → ?\n11. hammer → ?\n12. parrot → ?\n13. cherry → ?\n14. needle → ?\n15. lobster → ?\n\nClassify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n16. pencil → ?\n17. salmon → ?\n18. pepper → ?\n19. mirror → ?\n20. walnut → ?\n21. mango → ?\n22. eagle → ?\n23. orange → ?\n24. rabbit → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"early\", \"3\": \"early\", \"4\": \"late\", \"5\": \"late\", \"6\": \"object\", \"7\": \"animal\", \"8\": \"food\", \"9\": \"animal\", \"10\": \"object\", \"11\": \"object\", \"12\": \"animal\", \"13\": \"food\", \"14\": \"object\", \"15\": \"animal\", \"16\": \"2\", \"17\": \"2\", \"18\": \"2\", \"19\": \"2\", \"20\": \"2\", \"21\": \"2\", \"22\": \"2\", \"23\": \"2\", \"24\": \"2\"}}"
 },
 {
  "task_id": "shifting_frontier_035",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. falcon → ?\n2. rabbit → ?\n3. pencil → ?\n4. pepper → ?\n5. candle → ?\n\nClassify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. donkey → ?\n7. salmon → ?\n8. walnut → ?\n9. pigeon → ?\n10. parrot → ?\n11. cherry → ?\n12. basket → ?\n13. hammer → ?\n14. mango → ?\n15. shovel → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n16. wrench → ?\n17. turnip → ?\n18. banana → ?\n19. orange → ?\n20. tiger → ?\n21. mirror → ?\n22. needle → ?\n23. eagle → ?\n24. lobster → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"animal\", \"3\": \"object\", \"4\": \"food\", \"5\": \"object\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\", \"11\": \"2\", \"12\": \"2\", \"13\": \"2\", \"14\": \"2\", \"15\": \"2\", \"16\": \"late\", \"17\": \"late\", \"18\": \"early\", \"19\": \"late\", \"20\": \"late\", \"21\": \"early\", \"22\": \"late\", \"23\": \"early\", \"24\": \"early\"}}"
 },
 {
  "task_id": "shifting_frontier_036",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. salmon → ?\n2. falcon → ?\n3. tiger → ?\n4. pigeon → ?\n5. eagle → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n6. wrench → ?\n7. orange → ?\n8. donkey → ?\n9. parrot → ?\n10. cherry → ?\n11. pencil → ?\n12. walnut → ?\n13. pepper → ?\n14. basket → ?\n15. hammer → ?\n\nClassify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n16. candle → ?\n17. mango → ?\n18. needle → ?\n19. rabbit → ?\n20. banana → ?\n21. shovel → ?\n22. mirror → ?\n23. turnip → ?\n24. lobster → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"animal\", \"3\": \"animal\", \"4\": \"animal\", \"5\": \"animal\", \"6\": \"late\", \"7\": \"late\", \"8\": \"early\", \"9\": \"late\", \"10\": \"early\", \"11\": \"late\", \"12\": \"late\", \"13\": \"late\", \"14\": \"early\", \"15\": \"early\", \"16\": \"2\", \"17\": \"2\", \"18\": \"2\", \"19\": \"2\", \"20\": \"3+\", \"21\": \"2\", \"22\": \"2\", \"23\": \"2\", \"24\": \"2\"}}"
 },
 {
  "task_id": "shifting_frontier_037",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. mango → ?\n2. needle → ?\n3. shovel → ?\n4. salmon → ?\n5. candle → ?\n\nClassify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. lobster → ?\n7. basket → ?\n8. wrench → ?\n9. pencil → ?\n10. pepper → ?\n11. pigeon → ?\n12. hammer → ?\n13. cherry → ?\n14. parrot → ?\n15. rabbit → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n16. falcon → ?\n17. orange → ?\n18. tiger → ?\n19. banana → ?\n20. donkey → ?\n21. eagle → ?\n22. walnut → ?\n23. turnip → ?\n24. mirror → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"object\", \"3\": \"object\", \"4\": \"food\", \"5\": \"object\", \"6\": \"2\", \"7\": \"2\", \"8\": \"1\", \"9\": \"2\", \"10\": \"2\", \"11\": \"2\", \"12\": \"2\", \"13\": \"2\", \"14\": \"2\", \"15\": \"2\", \"16\": \"early\", \"17\": \"late\", \"18\": \"late\", \"19\": \"early\", \"20\": \"early\", \"21\": \"early\", \"22\": \"late\", \"23\": \"late\", \"24\": \"early\"}}"
 },
 {
  "task_id": "shifting_frontier_038",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. pepper → ?\n2. mango → ?\n3. banana → ?\n4. rabbit → ?\n5. pigeon → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n6. shovel → ?\n7. orange → ?\n8. wrench → ?\n9. candle → ?\n10. pencil → ?\n11. basket → ?\n12. walnut → ?\n13. hammer → ?\n14. needle → ?\n15. salmon → ?\n\nClassify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n16. cherry → ?\n17. falcon → ?\n18. lobster → ?\n19. parrot → ?\n20. mirror → ?\n21. turnip → ?\n22. donkey → ?\n23. eagle → ?\n24. tiger → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"food\", \"3\": \"food\", \"4\": \"animal\", \"5\": \"animal\", \"6\": \"late\", \"7\": \"late\", \"8\": \"late\", \"9\": \"early\", \"10\": \"late\", \"11\": \"early\", \"12\": \"late\", \"13\": \"early\", \"14\": \"late\", \"15\": \"late\", \"16\": \"2\", \"17\": \"2\", \"18\": \"2\", \"19\": \"2\", \"20\": \"2\", \"21\": \"2\", \"22\": \"2\", \"23\": \"2\", \"24\": \"2\"}}"
 },
 {
  "task_id": "shifting_frontier_039",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. donkey → ?\n2. hammer → ?\n3. walnut → ?\n4. pepper → ?\n5. banana → ?\n\nClassify each word by CATEGORY: animal, food, or object\n\n6. tiger → ?\n7. shovel → ?\n8. mirror → ?\n9. lobster → ?\n10. candle → ?\n11. wrench → ?\n12. basket → ?\n13. pigeon → ?\n14. turnip → ?\n15. eagle → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n16. mango → ?\n17. rabbit → ?\n18. orange → ?\n19. needle → ?\n20. cherry → ?\n21. salmon → ?\n22. parrot → ?\n23. pencil → ?\n24. falcon → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"2\", \"4\": \"2\", \"5\": \"3+\", \"6\": \"animal\", \"7\": \"object\", \"8\": \"object\", \"9\": \"animal\", \"10\": \"object\", \"11\": \"object\", \"12\": \"object\", \"13\": \"animal\", \"14\": \"food\", \"15\": \"animal\", \"16\": \"early\", \"17\": \"late\", \"18\": \"late\", \"19\": \"late\", \"20\": \"early\", \"21\": \"late\", \"22\": \"late\", \"23\": \"late\", \"24\": \"early\"}}"
 },
 {
  "task_id": "ior_easy_000",
  "task_type": "inhibition_return",
  "difficulty": "Easy",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe warehouse at the east wing received a shipment on Friday. It contained 60 crates of ceramics, each weighing 8 kilograms. The shipment was logged by Viktor under reference number WH-7120. The botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The total value of the shipment was $9,100.\n\n--- Passage B ---\nThe weather station at the valley floor recorded conditions on Tuesday. Temperature reached 18.5°C with humidity at 55%. Wind speed was measured at 15 km/h from the north. A street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Barometric pressure stood at 1021 hPa.\n\n--- Passage C ---\nExperiment EX-884 was conducted by Yara in lab C-05. The sample was heated to 45°C for 45 minutes. The resulting concentration was 45.1 mg/L. A shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The pH of the solution measured 5.8 at completion.\n\nAnswer each question:\nQ1. (About Passage C) What was the resulting concentration?\nQ2. (About Passage A) What was the reference number?\nQ3. (About Passage C) How many minutes was the heating duration?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"45.1\", \"2\": \"WH-7120\", \"3\": \"45\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the resulting concentration?\", \"answer_key\": \"concentration\", \"gold\": \"45.1\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-7120\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"How many minutes was the heating duration?\", \"answer_key\": \"duration\", \"gold\": \"45\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_easy_001",
  "task_type": "inhibition_return",
  "difficulty": "Easy",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-884 was conducted by Joaquin in lab C-05. The sample was heated to 120°C for 15 minutes. The resulting concentration was 45.1 mg/L. A shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The pH of the solution measured 4.5 at completion.\n\n--- Passage B ---\nThe warehouse at building 7 received a shipment on Friday. It contained 15 crates of electronics, each weighing 8 kilograms. The shipment was logged by Amara under reference number WH-7120. The workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The total value of the shipment was $9,100.\n\n--- Passage C ---\nThe weather station at the coastal station recorded conditions on Tuesday. Temperature reached 27.8°C with humidity at 45%. Wind speed was measured at 52 km/h from the west. A street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Barometric pressure stood at 1015 hPa.\n\nAnswer each question:\nQ1. (About Passage B) Who logged the shipment?\nQ2. (About Passage A) What was the resulting concentration?\nQ3. (About Passage B) How many crates were in the shipment?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Amara\", \"2\": \"45.1\", \"3\": \"15\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"Who logged the shipment?\", \"answer_key\": \"person\", \"gold\": \"Amara\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the resulting concentration?\", \"answer_key\": \"concentration\", \"gold\": \"45.1\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"How many crates were in the shipment?\", \"answer_key\": \"quantity\", \"gold\": \"15\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_easy_002",
  "task_type": "inhibition_return",
  "difficulty": "Easy",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the valley floor recorded conditions on Monday. Temperature reached 8.9°C with humidity at 62%. Wind speed was measured at 8 km/h from the west. A shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Barometric pressure stood at 997 hPa.\n\n--- Passage B ---\nExperiment EX-884 was conducted by Adaeze in lab A-18. The sample was heated to 45°C for 30 minutes. The resulting concentration was 28.7 mg/L. Construction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The pH of the solution measured 7.1 at completion.\n\n--- Passage C ---\nThe warehouse at the east wing received a shipment on Thursday. It contained 15 crates of textiles, each weighing 12 kilograms. The shipment was logged by Tariq under reference number WH-2293. The old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The total value of the shipment was $3,600.\n\nAnswer each question:\nQ1. (About Passage B) What temperature was the sample heated to?\nQ2. (About Passage C) What day did the shipment arrive?\nQ3. (About Passage B) Who conducted the experiment?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"45\", \"2\": \"Thursday\", \"3\": \"Adaeze\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What temperature was the sample heated to?\", \"answer_key\": \"temp\", \"gold\": \"45\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What day did the shipment arrive?\", \"answer_key\": \"day\", \"gold\": \"Thursday\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"Who conducted the experiment?\", \"answer_key\": \"person\", \"gold\": \"Adaeze\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_easy_003",
  "task_type": "inhibition_return",
  "difficulty": "Easy",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the airport recorded conditions on Friday. Temperature reached 8.9°C with humidity at 55%. Wind speed was measured at 42 km/h from the southwest. The market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Barometric pressure stood at 1015 hPa.\n\n--- Passage B ---\nExperiment EX-350 was conducted by Xander in lab B-12. The sample was heated to 80°C for 120 minutes. The resulting concentration was 8.9 mg/L. The market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The pH of the solution measured 6.2 at completion.\n\n--- Passage C ---\nThe warehouse at building 7 received a shipment on Friday. It contained 72 crates of textiles, each weighing 32 kilograms. The shipment was logged by Dariush under reference number WH-8841. Fog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The total value of the shipment was $9,100.\n\nAnswer each question:\nQ1. (About Passage B) What was the final pH?\nQ2. (About Passage A) What was the temperature?\nQ3. (About Passage B) How many minutes was the heating duration?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"6.2\", \"2\": \"8.9\", \"3\": \"120\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the final pH?\", \"answer_key\": \"ph\", \"gold\": \"6.2\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the temperature?\", \"answer_key\": \"temp\", \"gold\": \"8.9\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"How many minutes was the heating duration?\", \"answer_key\": \"duration\", \"gold\": \"120\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_easy_004",
  "task_type": "inhibition_return",
  "difficulty": "Easy",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-229 was conducted by Dariush in lab C-05. The sample was heated to 45°C for 15 minutes. The resulting concentration was 45.1 mg/L. The clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The pH of the solution measured 4.5 at completion.\n\n--- Passage B ---\nThe warehouse at building 7 received a shipment on Friday. It contained 60 crates of ceramics, each weighing 15 kilograms. The shipment was logged by Zain under reference number WH-4017. Construction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The total value of the shipment was $9,100.\n\n--- Passage C ---\nThe weather station at the coastal station recorded conditions on Friday. Temperature reached 23.2°C with humidity at 62%. Wind speed was measured at 52 km/h from the west. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Barometric pressure stood at 997 hPa.\n\nAnswer each question:\nQ1. (About Passage B) What day did the shipment arrive?\nQ2. (About Passage A) What was the resulting concentration?\nQ3. (About Passage B) What was the reference number?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Friday\", \"2\": \"45.1\", \"3\": \"WH-4017\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What day did the shipment arrive?\", \"answer_key\": \"day\", \"gold\": \"Friday\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the resulting concentration?\", \"answer_key\": \"concentration\", \"gold\": \"45.1\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-4017\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_easy_005",
  "task_type": "inhibition_return",
  "difficulty": "Easy",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-712 was conducted by Leif in lab B-12. The sample was heated to 65°C for 30 minutes. The resulting concentration was 33.6 mg/L. The workshop on Vesna Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The pH of the solution measured 6.2 at completion.\n\n--- Passage B ---\nThe warehouse at the east wing received a shipment on Tuesday. It contained 72 crates of ceramics, each weighing 25 kilograms. The shipment was logged by Priya under reference number WH-2293. The old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The total value of the shipment was $3,600.\n\n--- Passage C ---\nThe weather station at the coastal station recorded conditions on Wednesday. Temperature reached 23.2°C with humidity at 62%. Wind speed was measured at 15 km/h from the east. The market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Barometric pressure stood at 1015 hPa.\n\nAnswer each question:\nQ1. (About Passage A) What temperature was the sample heated to?\nQ2. (About Passage B) How much did each crate weigh?\nQ3. (About Passage A) What was the experiment ID?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"65\", \"2\": \"25\", \"3\": \"EX-712\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What temperature was the sample heated to?\", \"answer_key\": \"temp\", \"gold\": \"65\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"How much did each crate weigh?\", \"answer_key\": \"weight\", \"gold\": \"25\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the experiment ID?\", \"answer_key\": \"exp_id\", \"gold\": \"EX-712\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_easy_006",
  "task_type": "inhibition_return",
  "difficulty": "Easy",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe warehouse at the north dock received a shipment on Wednesday. It contained 72 crates of chemicals, each weighing 25 kilograms. The shipment was logged by Colette under reference number WH-2293. Construction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The total value of the shipment was $3,600.\n\n--- Passage B ---\nExperiment EX-884 was conducted by Priya in lab C-05. The sample was heated to 150°C for 45 minutes. The resulting concentration was 8.9 mg/L. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The pH of the solution measured 6.9 at completion.\n\n--- Passage C ---\nThe weather station at the airport recorded conditions on Tuesday. Temperature reached 18.5°C with humidity at 62%. Wind speed was measured at 42 km/h from the east. A thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Barometric pressure stood at 1008 hPa.\n\nAnswer each question:\nQ1. (About Passage B) What was the final pH?\nQ2. (About Passage C) What day were conditions recorded?\nQ3. (About Passage B) What temperature was the sample heated to?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"6.9\", \"2\": \"Tuesday\", \"3\": \"150\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the final pH?\", \"answer_key\": \"ph\", \"gold\": \"6.9\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What day were conditions recorded?\", \"answer_key\": \"day\", \"gold\": \"Tuesday\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What temperature was the sample heated to?\", \"answer_key\": \"temp\", \"gold\": \"150\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_easy_007",
  "task_type": "inhibition_return",
  "difficulty": "Easy",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe warehouse at the north dock received a shipment on Friday. It contained 36 crates of ceramics, each weighing 18 kilograms. The shipment was logged by Adaeze under reference number WH-2293. A thin rain began to fall just as Qadir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The total value of the shipment was $12,300.\n\n--- Passage B ---\nThe weather station at the valley floor recorded conditions on Tuesday. Temperature reached 23.2°C with humidity at 62%. Wind speed was measured at 15 km/h from the north. The library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nalini fully understood. The wooden shelves bowed slightly under the weight. Barometric pressure stood at 1021 hPa.\n\n--- Passage C ---\nExperiment EX-401 was conducted by Ravi in lab C-05. The sample was heated to 150°C for 45 minutes. The resulting concentration was 12.4 mg/L. The old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The pH of the solution measured 8.4 at completion.\n\nAnswer each question:\nQ1. (About Passage A) How much did each crate weigh?\nQ2. (About Passage C) How many minutes was the heating duration?\nQ3. (About Passage A) Who logged the shipment?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"18\", \"2\": \"45\", \"3\": \"Adaeze\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"How much did each crate weigh?\", \"answer_key\": \"weight\", \"gold\": \"18\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"How many minutes was the heating duration?\", \"answer_key\": \"duration\", \"gold\": \"45\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"Who logged the shipment?\", \"answer_key\": \"person\", \"gold\": \"Adaeze\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_medium_008",
  "task_type": "inhibition_return",
  "difficulty": "Medium",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-229 was conducted by Bashir in lab B-12. The sample was heated to 150°C for 90 minutes. The resulting concentration was 45.1 mg/L. The clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The pH of the solution measured 6.2 at completion.\n\nThe workshop on Yuki Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\n--- Passage B ---\nThe warehouse at the central depot received a shipment on Wednesday. It contained 24 crates of chemicals, each weighing 18 kilograms. The shipment was logged by Magnus under reference number WH-4017. The market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The total value of the shipment was $4,200.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\n--- Passage C ---\nThe weather station at the valley floor recorded conditions on Monday. Temperature reached 31.7°C with humidity at 55%. Wind speed was measured at 28 km/h from the east. The old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Barometric pressure stood at 1021 hPa.\n\nAnswer each question:\nQ1. (About Passage B) What was the total value?\nQ2. (About Passage C) What was the temperature?\nQ3. (About Passage B) How many crates were in the shipment?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"4,200\", \"2\": \"31.7\", \"3\": \"24\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the total value?\", \"answer_key\": \"value\", \"gold\": \"4,200\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the temperature?\", \"answer_key\": \"temp\", \"gold\": \"31.7\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"How many crates were in the shipment?\", \"answer_key\": \"quantity\", \"gold\": \"24\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_medium_009",
  "task_type": "inhibition_return",
  "difficulty": "Medium",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the coastal station recorded conditions on Friday. Temperature reached 18.5°C with humidity at 55%. Wind speed was measured at 52 km/h from the east. Trade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient. Barometric pressure stood at 1008 hPa.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\n--- Passage B ---\nThe warehouse at building 7 received a shipment on Monday. It contained 48 crates of textiles, each weighing 15 kilograms. The shipment was logged by Zora under reference number WH-8841. Trade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient. The total value of the shipment was $3,600.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n\n--- Passage C ---\nExperiment EX-884 was conducted by Qadir in lab A-18. The sample was heated to 45°C for 60 minutes. The resulting concentration was 33.6 mg/L. The market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The pH of the solution measured 6.2 at completion.\n\nAnswer each question:\nQ1. (About Passage C) What temperature was the sample heated to?\nQ2. (About Passage B) What day did the shipment arrive?\nQ3. (About Passage C) Who conducted the experiment?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"45\", \"2\": \"Monday\", \"3\": \"Qadir\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What temperature was the sample heated to?\", \"answer_key\": \"temp\", \"gold\": \"45\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What day did the shipment arrive?\", \"answer_key\": \"day\", \"gold\": \"Monday\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"Who conducted the experiment?\", \"answer_key\": \"person\", \"gold\": \"Qadir\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_medium_010",
  "task_type": "inhibition_return",
  "difficulty": "Medium",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-712 was conducted by Nico in lab C-05. The sample was heated to 65°C for 60 minutes. The resulting concentration was 28.7 mg/L. Trade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient. The pH of the solution measured 5.8 at completion.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\n--- Passage B ---\nThe warehouse at the north dock received a shipment on Thursday. It contained 72 crates of electronics, each weighing 18 kilograms. The shipment was logged by Celine under reference number WH-3506. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The total value of the shipment was $3,600.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\n--- Passage C ---\nThe weather station at the hilltop recorded conditions on Tuesday. Temperature reached 12.4°C with humidity at 87%. Wind speed was measured at 52 km/h from the southwest. The clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Barometric pressure stood at 997 hPa.\n\nAnswer each question:\nQ1. (About Passage B) What was the total value?\nQ2. (About Passage C) What was the humidity percentage?\nQ3. (About Passage B) Who logged the shipment?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"3,600\", \"2\": \"87\", \"3\": \"Celine\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the total value?\", \"answer_key\": \"value\", \"gold\": \"3,600\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the humidity percentage?\", \"answer_key\": \"humidity\", \"gold\": \"87\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"Who logged the shipment?\", \"answer_key\": \"person\", \"gold\": \"Celine\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_medium_011",
  "task_type": "inhibition_return",
  "difficulty": "Medium",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-229 was conducted by Haruto in lab A-18. The sample was heated to 80°C for 30 minutes. The resulting concentration was 33.6 mg/L. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The pH of the solution measured 5.8 at completion.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\n--- Passage B ---\nThe weather station at the hilltop recorded conditions on Monday. Temperature reached 27.8°C with humidity at 33%. Wind speed was measured at 28 km/h from the north. The clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Barometric pressure stood at 1013 hPa.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\n--- Passage C ---\nThe warehouse at building 7 received a shipment on Tuesday. It contained 24 crates of chemicals, each weighing 8 kilograms. The shipment was logged by Paloma under reference number WH-4017. A thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The total value of the shipment was $3,600.\n\nAnswer each question:\nQ1. (About Passage C) What was the total value?\nQ2. (About Passage B) What was the temperature?\nQ3. (About Passage C) What day did the shipment arrive?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"3,600\", \"2\": \"27.8\", \"3\": \"Tuesday\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the total value?\", \"answer_key\": \"value\", \"gold\": \"3,600\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the temperature?\", \"answer_key\": \"temp\", \"gold\": \"27.8\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What day did the shipment arrive?\", \"answer_key\": \"day\", \"gold\": \"Tuesday\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_medium_012",
  "task_type": "inhibition_return",
  "difficulty": "Medium",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the hilltop recorded conditions on Monday. Temperature reached 12.4°C with humidity at 55%. Wind speed was measured at 8 km/h from the south. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Barometric pressure stood at 997 hPa.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\n--- Passage B ---\nThe warehouse at the east wing received a shipment on Friday. It contained 48 crates of textiles, each weighing 8 kilograms. The shipment was logged by Joaquin under reference number WH-8841. Fog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The total value of the shipment was $9,100.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\n--- Passage C ---\nExperiment EX-350 was conducted by Nico in lab A-18. The sample was heated to 45°C for 15 minutes. The resulting concentration was 33.6 mg/L. Trade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient. The pH of the solution measured 6.2 at completion.\n\nAnswer each question:\nQ1. (About Passage A) What day were conditions recorded?\nQ2. (About Passage C) What was the final pH?\nQ3. (About Passage A) What was the temperature?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Monday\", \"2\": \"6.2\", \"3\": \"12.4\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What day were conditions recorded?\", \"answer_key\": \"day\", \"gold\": \"Monday\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the final pH?\", \"answer_key\": \"ph\", \"gold\": \"6.2\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the temperature?\", \"answer_key\": \"temp\", \"gold\": \"12.4\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_medium_013",
  "task_type": "inhibition_return",
  "difficulty": "Medium",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the valley floor recorded conditions on Wednesday. Temperature reached 23.2°C with humidity at 33%. Wind speed was measured at 28 km/h from the northeast. The botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Barometric pressure stood at 1015 hPa.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\n--- Passage B ---\nThe warehouse at the east wing received a shipment on Tuesday. It contained 15 crates of machinery parts, each weighing 32 kilograms. The shipment was logged by Xander under reference number WH-2293. The market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The total value of the shipment was $4,200.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\n--- Passage C ---\nExperiment EX-401 was conducted by Nalini in lab A-18. The sample was heated to 150°C for 15 minutes. The resulting concentration was 8.9 mg/L. The old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The pH of the solution measured 7.1 at completion.\n\nAnswer each question:\nQ1. (About Passage C) What was the resulting concentration?\nQ2. (About Passage B) What was the reference number?\nQ3. (About Passage C) Who conducted the experiment?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"8.9\", \"2\": \"WH-2293\", \"3\": \"Nalini\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the resulting concentration?\", \"answer_key\": \"concentration\", \"gold\": \"8.9\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-2293\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"Who conducted the experiment?\", \"answer_key\": \"person\", \"gold\": \"Nalini\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_medium_014",
  "task_type": "inhibition_return",
  "difficulty": "Medium",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-401 was conducted by Freya in lab C-05. The sample was heated to 120°C for 45 minutes. The resulting concentration was 33.6 mg/L. The clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The pH of the solution measured 4.5 at completion.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\n--- Passage B ---\nThe warehouse at building 7 received a shipment on Monday. It contained 72 crates of textiles, each weighing 25 kilograms. The shipment was logged by Xander under reference number WH-3506. The library held over thirty thousand volumes, most of them catalogued by hand in a system that only Priya fully understood. The wooden shelves bowed slightly under the weight. The total value of the shipment was $9,100.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\n--- Passage C ---\nThe weather station at the airport recorded conditions on Friday. Temperature reached 8.9°C with humidity at 55%. Wind speed was measured at 52 km/h from the south. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Barometric pressure stood at 1015 hPa.\n\nAnswer each question:\nQ1. (About Passage C) What was the humidity percentage?\nQ2. (About Passage A) How many minutes was the heating duration?\nQ3. (About Passage C) What was the wind speed?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"55\", \"2\": \"45\", \"3\": \"52\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the humidity percentage?\", \"answer_key\": \"humidity\", \"gold\": \"55\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"How many minutes was the heating duration?\", \"answer_key\": \"duration\", \"gold\": \"45\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the wind speed?\", \"answer_key\": \"wind\", \"gold\": \"52\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_medium_015",
  "task_type": "inhibition_return",
  "difficulty": "Medium",
  "prompt": "Read the following 3 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the valley floor recorded conditions on Wednesday. Temperature reached 27.8°C with humidity at 62%. Wind speed was measured at 42 km/h from the south. The library held over thirty thousand volumes, most of them catalogued by hand in a system that only Bram fully understood. The wooden shelves bowed slightly under the weight. Barometric pressure stood at 1003 hPa.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\n--- Passage B ---\nThe warehouse at building 7 received a shipment on Tuesday. It contained 15 crates of ceramics, each weighing 25 kilograms. The shipment was logged by Ines under reference number WH-4017. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The total value of the shipment was $7,850.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight.\n\n--- Passage C ---\nExperiment EX-229 was conducted by Priya in lab D-03. The sample was heated to 150°C for 15 minutes. The resulting concentration was 33.6 mg/L. The workshop on Willa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The pH of the solution measured 5.8 at completion.\n\nAnswer each question:\nQ1. (About Passage B) How much did each crate weigh?\nQ2. (About Passage A) What was the temperature?\nQ3. (About Passage B) What was the total value?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"25\", \"2\": \"27.8\", \"3\": \"7,850\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"How much did each crate weigh?\", \"answer_key\": \"weight\", \"gold\": \"25\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the temperature?\", \"answer_key\": \"temp\", \"gold\": \"27.8\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the total value?\", \"answer_key\": \"value\", \"gold\": \"7,850\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_hard_016",
  "task_type": "inhibition_return",
  "difficulty": "Hard",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe warehouse at the central depot received a shipment on Friday. It contained 48 crates of textiles, each weighing 8 kilograms. The shipment was logged by Lumi under reference number WH-3506. The workshop on Haruto Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The total value of the shipment was $4,200.\n\nThe workshop on Freya Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\n--- Passage B ---\nThe weather station at the airport recorded conditions on Wednesday. Temperature reached 27.8°C with humidity at 33%. Wind speed was measured at 15 km/h from the southwest. A shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Barometric pressure stood at 1003 hPa.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\n--- Passage C ---\nExperiment EX-350 was conducted by Elara in lab C-05. The sample was heated to 95°C for 90 minutes. The resulting concentration was 45.1 mg/L. The annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. The pH of the solution measured 7.1 at completion.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\n--- Passage D ---\nThe warehouse at the north dock received a shipment on Tuesday. It contained 60 crates of ceramics, each weighing 8 kilograms. The shipment was logged by Kaia under reference number WH-8841. A street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The total value of the shipment was $9,100.\n\nAnswer each question:\nQ1. (About Passage B) What day were conditions recorded?\nQ2. (About Passage A) What was the reference number?\nQ3. (About Passage D) What was the total value?\nQ4. (About Passage B) What was the temperature?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Wednesday\", \"2\": \"WH-3506\", \"3\": \"9,100\", \"4\": \"27.8\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What day were conditions recorded?\", \"answer_key\": \"day\", \"gold\": \"Wednesday\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-3506\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What was the total value?\", \"answer_key\": \"value\", \"gold\": \"9,100\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the temperature?\", \"answer_key\": \"temp\", \"gold\": \"27.8\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_hard_017",
  "task_type": "inhibition_return",
  "difficulty": "Hard",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the airport recorded conditions on Tuesday. Temperature reached 18.5°C with humidity at 62%. Wind speed was measured at 52 km/h from the southwest. A thin rain began to fall just as Willa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Barometric pressure stood at 1008 hPa.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\n--- Passage B ---\nExperiment EX-884 was conducted by Ines in lab B-12. The sample was heated to 80°C for 30 minutes. The resulting concentration was 8.9 mg/L. The old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The pH of the solution measured 4.5 at completion.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\n--- Passage C ---\nThe warehouse at building 7 received a shipment on Tuesday. It contained 48 crates of electronics, each weighing 32 kilograms. The shipment was logged by Celine under reference number WH-2293. The botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The total value of the shipment was $3,600.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\n--- Passage D ---\nExperiment EX-712 was conducted by Runa in lab B-12. The sample was heated to 45°C for 45 minutes. The resulting concentration was 12.4 mg/L. A street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The pH of the solution measured 6.9 at completion.\n\nAnswer each question:\nQ1. (About Passage C) What was the total value?\nQ2. (About Passage B) What was the final pH?\nQ3. (About Passage D) What was the resulting concentration?\nQ4. (About Passage C) What was the reference number?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"3,600\", \"2\": \"4.5\", \"3\": \"12.4\", \"4\": \"WH-2293\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the total value?\", \"answer_key\": \"value\", \"gold\": \"3,600\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the final pH?\", \"answer_key\": \"ph\", \"gold\": \"4.5\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What was the resulting concentration?\", \"answer_key\": \"concentration\", \"gold\": \"12.4\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-2293\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_hard_018",
  "task_type": "inhibition_return",
  "difficulty": "Hard",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe warehouse at the central depot received a shipment on Thursday. It contained 48 crates of textiles, each weighing 8 kilograms. The shipment was logged by Soren under reference number WH-4017. Fog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The total value of the shipment was $4,200.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Maren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\n--- Passage B ---\nThe weather station at the airport recorded conditions on Tuesday. Temperature reached 23.2°C with humidity at 33%. Wind speed was measured at 42 km/h from the east. The workshop on Uma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Barometric pressure stood at 1021 hPa.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Haruto fully understood. The wooden shelves bowed slightly under the weight.\n\n--- Passage C ---\nExperiment EX-350 was conducted by Joelle in lab B-12. The sample was heated to 80°C for 120 minutes. The resulting concentration was 28.7 mg/L. Trade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient. The pH of the solution measured 4.5 at completion.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe workshop on Bashir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\n--- Passage D ---\nThe weather station at the hilltop recorded conditions on Friday. Temperature reached 27.8°C with humidity at 62%. Wind speed was measured at 8 km/h from the northeast. Evening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Barometric pressure stood at 1003 hPa.\n\nAnswer each question:\nQ1. (About Passage D) What was the humidity percentage?\nQ2. (About Passage B) What was the barometric pressure?\nQ3. (About Passage A) What was the reference number?\nQ4. (About Passage D) What was the barometric pressure?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"62\", \"2\": \"1021\", \"3\": \"WH-4017\", \"4\": \"1003\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What was the humidity percentage?\", \"answer_key\": \"humidity\", \"gold\": \"62\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the barometric pressure?\", \"answer_key\": \"pressure\", \"gold\": \"1021\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-4017\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What was the barometric pressure?\", \"answer_key\": \"pressure\", \"gold\": \"1003\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_hard_019",
  "task_type": "inhibition_return",
  "difficulty": "Hard",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the airport recorded conditions on Monday. Temperature reached 31.7°C with humidity at 55%. Wind speed was measured at 35 km/h from the west. A street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Barometric pressure stood at 1003 hPa.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\n--- Passage B ---\nExperiment EX-350 was conducted by Joelle in lab A-18. The sample was heated to 45°C for 90 minutes. The resulting concentration was 56.2 mg/L. The workshop on Tariq Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The pH of the solution measured 8.4 at completion.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\n--- Passage C ---\nThe warehouse at the north dock received a shipment on Wednesday. It contained 15 crates of textiles, each weighing 15 kilograms. The shipment was logged by Ines under reference number WH-7120. Fog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The total value of the shipment was $4,200.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Zora Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\n--- Passage D ---\nThe weather station at the hilltop recorded conditions on Thursday. Temperature reached 31.7°C with humidity at 55%. Wind speed was measured at 15 km/h from the north. The market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Barometric pressure stood at 997 hPa.\n\nAnswer each question:\nQ1. (About Passage C) How many crates were in the shipment?\nQ2. (About Passage D) What direction was the wind from?\nQ3. (About Passage B) What temperature was the sample heated to?\nQ4. (About Passage C) What was the reference number?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"15\", \"2\": \"north\", \"3\": \"45\", \"4\": \"WH-7120\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"How many crates were in the shipment?\", \"answer_key\": \"quantity\", \"gold\": \"15\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What direction was the wind from?\", \"answer_key\": \"direction\", \"gold\": \"north\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What temperature was the sample heated to?\", \"answer_key\": \"temp\", \"gold\": \"45\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-7120\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_hard_020",
  "task_type": "inhibition_return",
  "difficulty": "Hard",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe warehouse at building 7 received a shipment on Wednesday. It contained 72 crates of ceramics, each weighing 12 kilograms. The shipment was logged by Ines under reference number WH-8841. The old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The total value of the shipment was $9,100.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\n--- Passage B ---\nExperiment EX-350 was conducted by Paloma in lab C-05. The sample was heated to 120°C for 45 minutes. The resulting concentration was 33.6 mg/L. Trade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. The pH of the solution measured 7.1 at completion.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\n--- Passage C ---\nThe weather station at the hilltop recorded conditions on Thursday. Temperature reached 8.9°C with humidity at 78%. Wind speed was measured at 15 km/h from the north. The botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Barometric pressure stood at 1008 hPa.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\n--- Passage D ---\nThe weather station at the hilltop recorded conditions on Friday. Temperature reached 27.8°C with humidity at 55%. Wind speed was measured at 52 km/h from the north. The market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Barometric pressure stood at 1008 hPa.\n\nAnswer each question:\nQ1. (About Passage B) How many minutes was the heating duration?\nQ2. (About Passage A) How many crates were in the shipment?\nQ3. (About Passage D) What was the wind speed?\nQ4. (About Passage B) What was the resulting concentration?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"45\", \"2\": \"72\", \"3\": \"52\", \"4\": \"33.6\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"How many minutes was the heating duration?\", \"answer_key\": \"duration\", \"gold\": \"45\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"How many crates were in the shipment?\", \"answer_key\": \"quantity\", \"gold\": \"72\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What was the wind speed?\", \"answer_key\": \"wind\", \"gold\": \"52\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the resulting concentration?\", \"answer_key\": \"concentration\", \"gold\": \"33.6\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_hard_021",
  "task_type": "inhibition_return",
  "difficulty": "Hard",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-350 was conducted by Haruto in lab C-05. The sample was heated to 80°C for 120 minutes. The resulting concentration was 33.6 mg/L. Trade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient. The pH of the solution measured 5.8 at completion.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Bram Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\n--- Passage B ---\nThe weather station at the coastal station recorded conditions on Thursday. Temperature reached 18.5°C with humidity at 45%. Wind speed was measured at 28 km/h from the southwest. Trade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient. Barometric pressure stood at 1015 hPa.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Magnus Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\n--- Passage C ---\nThe warehouse at building 7 received a shipment on Tuesday. It contained 60 crates of machinery parts, each weighing 8 kilograms. The shipment was logged by Runa under reference number WH-2293. Evening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The total value of the shipment was $15,400.\n\nThe workshop on Elara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\n--- Passage D ---\nThe warehouse at the east wing received a shipment on Thursday. It contained 36 crates of chemicals, each weighing 25 kilograms. The shipment was logged by Leif under reference number WH-4017. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The total value of the shipment was $12,300.\n\nAnswer each question:\nQ1. (About Passage A) What temperature was the sample heated to?\nQ2. (About Passage D) How many crates were in the shipment?\nQ3. (About Passage C) What was the reference number?\nQ4. (About Passage A) What was the experiment ID?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"80\", \"2\": \"36\", \"3\": \"WH-2293\", \"4\": \"EX-350\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What temperature was the sample heated to?\", \"answer_key\": \"temp\", \"gold\": \"80\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"How many crates were in the shipment?\", \"answer_key\": \"quantity\", \"gold\": \"36\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-2293\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the experiment ID?\", \"answer_key\": \"exp_id\", \"gold\": \"EX-350\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_hard_022",
  "task_type": "inhibition_return",
  "difficulty": "Hard",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-401 was conducted by Zain in lab C-05. The sample was heated to 95°C for 30 minutes. The resulting concentration was 28.7 mg/L. The market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The pH of the solution measured 6.2 at completion.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nalini fully understood. The wooden shelves bowed slightly under the weight.\n\n--- Passage B ---\nThe weather station at the coastal station recorded conditions on Friday. Temperature reached 12.4°C with humidity at 78%. Wind speed was measured at 42 km/h from the west. A shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Barometric pressure stood at 1013 hPa.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Priya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\n--- Passage C ---\nThe warehouse at the east wing received a shipment on Tuesday. It contained 24 crates of machinery parts, each weighing 15 kilograms. The shipment was logged by Dmitri under reference number WH-8841. A street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The total value of the shipment was $12,300.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\n--- Passage D ---\nThe weather station at the hilltop recorded conditions on Friday. Temperature reached 31.7°C with humidity at 62%. Wind speed was measured at 15 km/h from the southwest. A shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Barometric pressure stood at 1008 hPa.\n\nAnswer each question:\nQ1. (About Passage C) How much did each crate weigh?\nQ2. (About Passage A) What was the resulting concentration?\nQ3. (About Passage B) What was the wind speed?\nQ4. (About Passage C) What was the reference number?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"15\", \"2\": \"28.7\", \"3\": \"42\", \"4\": \"WH-8841\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"How much did each crate weigh?\", \"answer_key\": \"weight\", \"gold\": \"15\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the resulting concentration?\", \"answer_key\": \"concentration\", \"gold\": \"28.7\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the wind speed?\", \"answer_key\": \"wind\", \"gold\": \"42\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-8841\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_hard_023",
  "task_type": "inhibition_return",
  "difficulty": "Hard",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-229 was conducted by Colette in lab B-22. The sample was heated to 120°C for 45 minutes. The resulting concentration was 56.2 mg/L. The library held over thirty thousand volumes, most of them catalogued by hand in a system that only Runa fully understood. The wooden shelves bowed slightly under the weight. The pH of the solution measured 7.1 at completion.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\n--- Passage B ---\nThe warehouse at the central depot received a shipment on Wednesday. It contained 60 crates of machinery parts, each weighing 32 kilograms. The shipment was logged by Hana under reference number WH-3506. Evening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The total value of the shipment was $12,300.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\n--- Passage C ---\nThe weather station at the hilltop recorded conditions on Thursday. Temperature reached 31.7°C with humidity at 62%. Wind speed was measured at 35 km/h from the southwest. A street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Barometric pressure stood at 1013 hPa.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Joelle Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\n--- Passage D ---\nThe warehouse at building 7 received a shipment on Monday. It contained 36 crates of textiles, each weighing 25 kilograms. The shipment was logged by Amara under reference number WH-2293. The library held over thirty thousand volumes, most of them catalogued by hand in a system that only Runa fully understood. The wooden shelves bowed slightly under the weight. The total value of the shipment was $3,600.\n\nAnswer each question:\nQ1. (About Passage D) What was the reference number?\nQ2. (About Passage C) What was the wind speed?\nQ3. (About Passage B) What day did the shipment arrive?\nQ4. (About Passage D) What was the total value?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"WH-2293\", \"2\": \"35\", \"3\": \"Wednesday\", \"4\": \"3,600\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-2293\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the wind speed?\", \"answer_key\": \"wind\", \"gold\": \"35\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What day did the shipment arrive?\", \"answer_key\": \"day\", \"gold\": \"Wednesday\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What was the total value?\", \"answer_key\": \"value\", \"gold\": \"3,600\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_expert_024",
  "task_type": "inhibition_return",
  "difficulty": "Expert",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the airport recorded conditions on Tuesday. Temperature reached 31.7°C with humidity at 62%. Wind speed was measured at 8 km/h from the southwest. Evening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Barometric pressure stood at 1013 hPa.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\n--- Passage B ---\nExperiment EX-712 was conducted by Nalini in lab C-05. The sample was heated to 150°C for 30 minutes. The resulting concentration was 8.9 mg/L. A thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The pH of the solution measured 4.5 at completion.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\n--- Passage C ---\nThe warehouse at the east wing received a shipment on Friday. It contained 60 crates of textiles, each weighing 32 kilograms. The shipment was logged by Yara under reference number WH-8841. Trade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient. The total value of the shipment was $7,850.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Ravi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\n--- Passage D ---\nThe weather station at the hilltop recorded conditions on Friday. Temperature reached 12.4°C with humidity at 87%. Wind speed was measured at 15 km/h from the east. A shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Barometric pressure stood at 1015 hPa.\n\nAnswer each question:\nQ1. (About Passage A) What was the humidity percentage?\nQ2. (About Passage C) How many crates were in the shipment?\nQ3. (About Passage B) How many minutes was the heating duration?\nQ4. (About Passage A) What day were conditions recorded?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"62\", \"2\": \"60\", \"3\": \"30\", \"4\": \"Tuesday\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the humidity percentage?\", \"answer_key\": \"humidity\", \"gold\": \"62\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"How many crates were in the shipment?\", \"answer_key\": \"quantity\", \"gold\": \"60\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"How many minutes was the heating duration?\", \"answer_key\": \"duration\", \"gold\": \"30\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What day were conditions recorded?\", \"answer_key\": \"day\", \"gold\": \"Tuesday\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_expert_025",
  "task_type": "inhibition_return",
  "difficulty": "Expert",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the valley floor recorded conditions on Wednesday. Temperature reached 27.8°C with humidity at 55%. Wind speed was measured at 28 km/h from the east. Trade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. Barometric pressure stood at 1021 hPa.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\n--- Passage B ---\nExperiment EX-712 was conducted by Leif in lab D-03. The sample was heated to 65°C for 15 minutes. The resulting concentration was 28.7 mg/L. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The pH of the solution measured 6.2 at completion.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Bram fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\n--- Passage C ---\nThe warehouse at the north dock received a shipment on Thursday. It contained 48 crates of chemicals, each weighing 12 kilograms. The shipment was logged by Freya under reference number WH-7120. The botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The total value of the shipment was $7,850.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\n--- Passage D ---\nThe warehouse at building 7 received a shipment on Wednesday. It contained 60 crates of ceramics, each weighing 18 kilograms. The shipment was logged by Tariq under reference number WH-2293. The market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The total value of the shipment was $12,300.\n\nAnswer each question:\nQ1. (About Passage A) What was the barometric pressure?\nQ2. (About Passage C) How many crates were in the shipment?\nQ3. (About Passage B) How many minutes was the heating duration?\nQ4. (About Passage A) What was the temperature?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1021\", \"2\": \"48\", \"3\": \"15\", \"4\": \"27.8\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the barometric pressure?\", \"answer_key\": \"pressure\", \"gold\": \"1021\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"How many crates were in the shipment?\", \"answer_key\": \"quantity\", \"gold\": \"48\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"How many minutes was the heating duration?\", \"answer_key\": \"duration\", \"gold\": \"15\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the temperature?\", \"answer_key\": \"temp\", \"gold\": \"27.8\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_expert_026",
  "task_type": "inhibition_return",
  "difficulty": "Expert",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-712 was conducted by Ines in lab C-05. The sample was heated to 45°C for 120 minutes. The resulting concentration was 56.2 mg/L. Construction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The pH of the solution measured 5.8 at completion.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Greta fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\n--- Passage B ---\nThe weather station at the hilltop recorded conditions on Friday. Temperature reached 12.4°C with humidity at 78%. Wind speed was measured at 52 km/h from the east. The workshop on Bashir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Barometric pressure stood at 1021 hPa.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Willa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Willa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\n--- Passage C ---\nThe warehouse at the east wing received a shipment on Monday. It contained 48 crates of chemicals, each weighing 8 kilograms. The shipment was logged by Tala under reference number WH-3506. The annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. The total value of the shipment was $9,100.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\n--- Passage D ---\nThe weather station at the hilltop recorded conditions on Thursday. Temperature reached 23.2°C with humidity at 87%. Wind speed was measured at 28 km/h from the north. The library held over thirty thousand volumes, most of them catalogued by hand in a system that only Viktor fully understood. The wooden shelves bowed slightly under the weight. Barometric pressure stood at 1015 hPa.\n\nAnswer each question:\nQ1. (About Passage C) What was the total value?\nQ2. (About Passage A) Who conducted the experiment?\nQ3. (About Passage B) What was the temperature?\nQ4. (About Passage C) How many crates were in the shipment?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"9,100\", \"2\": \"Ines\", \"3\": \"12.4\", \"4\": \"48\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the total value?\", \"answer_key\": \"value\", \"gold\": \"9,100\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"Who conducted the experiment?\", \"answer_key\": \"person\", \"gold\": \"Ines\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the temperature?\", \"answer_key\": \"temp\", \"gold\": \"12.4\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"How many crates were in the shipment?\", \"answer_key\": \"quantity\", \"gold\": \"48\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_expert_027",
  "task_type": "inhibition_return",
  "difficulty": "Expert",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-350 was conducted by Ines in lab B-22. The sample was heated to 150°C for 60 minutes. The resulting concentration was 12.4 mg/L. The old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The pH of the solution measured 6.9 at completion.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\n--- Passage B ---\nThe warehouse at the north dock received a shipment on Friday. It contained 72 crates of chemicals, each weighing 18 kilograms. The shipment was logged by Paloma under reference number WH-7120. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The total value of the shipment was $12,300.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Yuki reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\n--- Passage C ---\nThe weather station at the airport recorded conditions on Monday. Temperature reached 31.7°C with humidity at 87%. Wind speed was measured at 35 km/h from the northeast. The clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Barometric pressure stood at 1013 hPa.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Viktor fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\n--- Passage D ---\nExperiment EX-401 was conducted by Magnus in lab D-03. The sample was heated to 150°C for 120 minutes. The resulting concentration was 8.9 mg/L. Evening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The pH of the solution measured 6.9 at completion.\n\nAnswer each question:\nQ1. (About Passage B) Who logged the shipment?\nQ2. (About Passage D) Who conducted the experiment?\nQ3. (About Passage D) Who conducted the experiment?\nQ4. (About Passage B) What was the reference number?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Paloma\", \"2\": \"Magnus\", \"3\": \"Magnus\", \"4\": \"WH-7120\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"Who logged the shipment?\", \"answer_key\": \"person\", \"gold\": \"Paloma\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"Who conducted the experiment?\", \"answer_key\": \"person\", \"gold\": \"Magnus\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"Who conducted the experiment?\", \"answer_key\": \"person\", \"gold\": \"Magnus\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-7120\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_expert_028",
  "task_type": "inhibition_return",
  "difficulty": "Expert",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the hilltop recorded conditions on Tuesday. Temperature reached 8.9°C with humidity at 45%. Wind speed was measured at 42 km/h from the south. A street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Barometric pressure stood at 1013 hPa.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Idris fully understood. The wooden shelves bowed slightly under the weight.\n\n--- Passage B ---\nThe warehouse at the north dock received a shipment on Friday. It contained 48 crates of machinery parts, each weighing 18 kilograms. The shipment was logged by Femi under reference number WH-8841. A shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The total value of the shipment was $12,300.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\n--- Passage C ---\nExperiment EX-884 was conducted by Haruto in lab A-18. The sample was heated to 45°C for 60 minutes. The resulting concentration was 45.1 mg/L. The market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The pH of the solution measured 6.2 at completion.\n\nA thin rain began to fall just as Haruto reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe workshop on Haruto Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\n--- Passage D ---\nThe weather station at the airport recorded conditions on Friday. Temperature reached 31.7°C with humidity at 62%. Wind speed was measured at 42 km/h from the southwest. Fog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Barometric pressure stood at 1013 hPa.\n\nAnswer each question:\nQ1. (About Passage B) What was the total value?\nQ2. (About Passage C) What temperature was the sample heated to?\nQ3. (About Passage A) What was the wind speed?\nQ4. (About Passage B) Who logged the shipment?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"12,300\", \"2\": \"45\", \"3\": \"42\", \"4\": \"Femi\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the total value?\", \"answer_key\": \"value\", \"gold\": \"12,300\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What temperature was the sample heated to?\", \"answer_key\": \"temp\", \"gold\": \"45\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the wind speed?\", \"answer_key\": \"wind\", \"gold\": \"42\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"Who logged the shipment?\", \"answer_key\": \"person\", \"gold\": \"Femi\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_expert_029",
  "task_type": "inhibition_return",
  "difficulty": "Expert",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the coastal station recorded conditions on Monday. Temperature reached 8.9°C with humidity at 78%. Wind speed was measured at 28 km/h from the north. The botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Barometric pressure stood at 1013 hPa.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\n--- Passage B ---\nThe warehouse at the central depot received a shipment on Friday. It contained 60 crates of ceramics, each weighing 8 kilograms. The shipment was logged by Nico under reference number WH-2293. A thin rain began to fall just as Adaeze reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The total value of the shipment was $3,600.\n\nA thin rain began to fall just as Ines reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\n--- Passage C ---\nExperiment EX-350 was conducted by Tala in lab C-05. The sample was heated to 80°C for 90 minutes. The resulting concentration was 8.9 mg/L. The annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. The pH of the solution measured 6.2 at completion.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\n--- Passage D ---\nThe warehouse at building 7 received a shipment on Friday. It contained 24 crates of machinery parts, each weighing 18 kilograms. The shipment was logged by Wren under reference number WH-3506. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The total value of the shipment was $3,600.\n\nAnswer each question:\nQ1. (About Passage C) Who conducted the experiment?\nQ2. (About Passage A) What was the wind speed?\nQ3. (About Passage B) How much did each crate weigh?\nQ4. (About Passage C) What was the final pH?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Tala\", \"2\": \"28\", \"3\": \"8\", \"4\": \"6.2\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"Who conducted the experiment?\", \"answer_key\": \"person\", \"gold\": \"Tala\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the wind speed?\", \"answer_key\": \"wind\", \"gold\": \"28\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"How much did each crate weigh?\", \"answer_key\": \"weight\", \"gold\": \"8\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the final pH?\", \"answer_key\": \"ph\", \"gold\": \"6.2\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_expert_030",
  "task_type": "inhibition_return",
  "difficulty": "Expert",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-350 was conducted by Celine in lab A-18. The sample was heated to 120°C for 60 minutes. The resulting concentration was 28.7 mg/L. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The pH of the solution measured 5.8 at completion.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\n--- Passage B ---\nThe warehouse at the central depot received a shipment on Wednesday. It contained 48 crates of electronics, each weighing 15 kilograms. The shipment was logged by Haruto under reference number WH-8841. The library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ravi fully understood. The wooden shelves bowed slightly under the weight. The total value of the shipment was $4,200.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\n--- Passage C ---\nThe weather station at the hilltop recorded conditions on Thursday. Temperature reached 31.7°C with humidity at 78%. Wind speed was measured at 28 km/h from the northeast. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Barometric pressure stood at 1013 hPa.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Ines Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\n--- Passage D ---\nThe warehouse at building 7 received a shipment on Monday. It contained 15 crates of machinery parts, each weighing 18 kilograms. The shipment was logged by Sigrid under reference number WH-2293. Evening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The total value of the shipment was $12,300.\n\nAnswer each question:\nQ1. (About Passage B) What day did the shipment arrive?\nQ2. (About Passage A) What was the experiment ID?\nQ3. (About Passage C) What day were conditions recorded?\nQ4. (About Passage B) What was the total value?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Wednesday\", \"2\": \"EX-350\", \"3\": \"Thursday\", \"4\": \"4,200\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What day did the shipment arrive?\", \"answer_key\": \"day\", \"gold\": \"Wednesday\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the experiment ID?\", \"answer_key\": \"exp_id\", \"gold\": \"EX-350\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What day were conditions recorded?\", \"answer_key\": \"day\", \"gold\": \"Thursday\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the total value?\", \"answer_key\": \"value\", \"gold\": \"4,200\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_expert_031",
  "task_type": "inhibition_return",
  "difficulty": "Expert",
  "prompt": "Read the following 4 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the airport recorded conditions on Monday. Temperature reached 12.4°C with humidity at 78%. Wind speed was measured at 52 km/h from the south. The workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Barometric pressure stood at 1013 hPa.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\n--- Passage B ---\nExperiment EX-350 was conducted by Zora in lab D-03. The sample was heated to 65°C for 45 minutes. The resulting concentration was 28.7 mg/L. Trade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient. The pH of the solution measured 6.9 at completion.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\n--- Passage C ---\nThe warehouse at building 7 received a shipment on Tuesday. It contained 72 crates of machinery parts, each weighing 32 kilograms. The shipment was logged by Runa under reference number WH-4017. Construction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The total value of the shipment was $4,200.\n\nA thin rain began to fall just as Colette reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Vesna reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\n--- Passage D ---\nExperiment EX-350 was conducted by Lumi in lab D-03. The sample was heated to 150°C for 45 minutes. The resulting concentration was 45.1 mg/L. The clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The pH of the solution measured 6.9 at completion.\n\nAnswer each question:\nQ1. (About Passage A) What was the temperature?\nQ2. (About Passage D) What was the resulting concentration?\nQ3. (About Passage C) How much did each crate weigh?\nQ4. (About Passage A) What was the wind speed?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"12.4\", \"2\": \"45.1\", \"3\": \"32\", \"4\": \"52\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the temperature?\", \"answer_key\": \"temp\", \"gold\": \"12.4\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What was the resulting concentration?\", \"answer_key\": \"concentration\", \"gold\": \"45.1\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"How much did each crate weigh?\", \"answer_key\": \"weight\", \"gold\": \"32\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the wind speed?\", \"answer_key\": \"wind\", \"gold\": \"52\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_frontier_032",
  "task_type": "inhibition_return",
  "difficulty": "Frontier",
  "prompt": "Read the following 5 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe warehouse at the east wing received a shipment on Wednesday. It contained 72 crates of ceramics, each weighing 18 kilograms. The shipment was logged by Greta under reference number WH-2293. Construction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The total value of the shipment was $9,100.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Tariq reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\n--- Passage B ---\nExperiment EX-712 was conducted by Zora in lab D-03. The sample was heated to 95°C for 45 minutes. The resulting concentration was 56.2 mg/L. Trade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient. The pH of the solution measured 4.5 at completion.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\n--- Passage C ---\nThe weather station at the coastal station recorded conditions on Friday. Temperature reached 31.7°C with humidity at 55%. Wind speed was measured at 28 km/h from the east. The annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Barometric pressure stood at 1015 hPa.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\n--- Passage D ---\nExperiment EX-884 was conducted by Zora in lab A-18. The sample was heated to 150°C for 60 minutes. The resulting concentration was 33.6 mg/L. Construction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The pH of the solution measured 6.9 at completion.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\n--- Passage E ---\nThe weather station at the airport recorded conditions on Monday. Temperature reached 18.5°C with humidity at 87%. Wind speed was measured at 15 km/h from the south. Construction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Barometric pressure stood at 1015 hPa.\n\nAnswer each question:\nQ1. (About Passage B) How many minutes was the heating duration?\nQ2. (About Passage C) What was the barometric pressure?\nQ3. (About Passage C) What was the wind speed?\nQ4. (About Passage D) What temperature was the sample heated to?\nQ5. (About Passage B) What was the experiment ID?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"45\", \"2\": \"1015\", \"3\": \"28\", \"4\": \"150\", \"5\": \"EX-712\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"How many minutes was the heating duration?\", \"answer_key\": \"duration\", \"gold\": \"45\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the barometric pressure?\", \"answer_key\": \"pressure\", \"gold\": \"1015\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the wind speed?\", \"answer_key\": \"wind\", \"gold\": \"28\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What temperature was the sample heated to?\", \"answer_key\": \"temp\", \"gold\": \"150\", \"phase\": \"shift\"}, {\"question_num\": 5, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the experiment ID?\", \"answer_key\": \"exp_id\", \"gold\": \"EX-712\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_frontier_033",
  "task_type": "inhibition_return",
  "difficulty": "Frontier",
  "prompt": "Read the following 5 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the hilltop recorded conditions on Wednesday. Temperature reached 12.4°C with humidity at 87%. Wind speed was measured at 28 km/h from the southwest. The clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Barometric pressure stood at 1013 hPa.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\n--- Passage B ---\nThe warehouse at the east wing received a shipment on Thursday. It contained 24 crates of ceramics, each weighing 8 kilograms. The shipment was logged by Paloma under reference number WH-4017. The clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The total value of the shipment was $4,200.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\n--- Passage C ---\nExperiment EX-712 was conducted by Ines in lab D-03. The sample was heated to 120°C for 120 minutes. The resulting concentration was 56.2 mg/L. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The pH of the solution measured 7.1 at completion.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\n--- Passage D ---\nThe weather station at the hilltop recorded conditions on Thursday. Temperature reached 31.7°C with humidity at 33%. Wind speed was measured at 35 km/h from the north. A shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Barometric pressure stood at 1013 hPa.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Runa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ines fully understood. The wooden shelves bowed slightly under the weight.\n\n--- Passage E ---\nThe weather station at the coastal station recorded conditions on Wednesday. Temperature reached 18.5°C with humidity at 62%. Wind speed was measured at 35 km/h from the northeast. The old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Barometric pressure stood at 1015 hPa.\n\nAnswer each question:\nQ1. (About Passage E) What was the humidity percentage?\nQ2. (About Passage D) What was the wind speed?\nQ3. (About Passage B) How many crates were in the shipment?\nQ4. (About Passage B) What was the total value?\nQ5. (About Passage E) What was the barometric pressure?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"62\", \"2\": \"35\", \"3\": \"24\", \"4\": \"4,200\", \"5\": \"1015\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"E\", \"passage_idx\": 4, \"question\": \"What was the humidity percentage?\", \"answer_key\": \"humidity\", \"gold\": \"62\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What was the wind speed?\", \"answer_key\": \"wind\", \"gold\": \"35\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"How many crates were in the shipment?\", \"answer_key\": \"quantity\", \"gold\": \"24\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the total value?\", \"answer_key\": \"value\", \"gold\": \"4,200\", \"phase\": \"shift\"}, {\"question_num\": 5, \"passage\": \"E\", \"passage_idx\": 4, \"question\": \"What was the barometric pressure?\", \"answer_key\": \"pressure\", \"gold\": \"1015\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_frontier_034",
  "task_type": "inhibition_return",
  "difficulty": "Frontier",
  "prompt": "Read the following 5 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-229 was conducted by Tala in lab C-05. The sample was heated to 120°C for 60 minutes. The resulting concentration was 45.1 mg/L. The clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The pH of the solution measured 8.4 at completion.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe workshop on Dmitri Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\n--- Passage B ---\nThe warehouse at the east wing received a shipment on Friday. It contained 72 crates of ceramics, each weighing 25 kilograms. The shipment was logged by Lumi under reference number WH-4017. The clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The total value of the shipment was $12,300.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\n--- Passage C ---\nThe weather station at the valley floor recorded conditions on Tuesday. Temperature reached 23.2°C with humidity at 33%. Wind speed was measured at 8 km/h from the northeast. Construction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Barometric pressure stood at 1015 hPa.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\n--- Passage D ---\nThe weather station at the hilltop recorded conditions on Monday. Temperature reached 12.4°C with humidity at 62%. Wind speed was measured at 42 km/h from the southwest. The ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Barometric pressure stood at 997 hPa.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Elio reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\n--- Passage E ---\nThe weather station at the valley floor recorded conditions on Monday. Temperature reached 23.2°C with humidity at 78%. Wind speed was measured at 52 km/h from the east. A shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Barometric pressure stood at 1013 hPa.\n\nAnswer each question:\nQ1. (About Passage E) What day were conditions recorded?\nQ2. (About Passage B) Who logged the shipment?\nQ3. (About Passage A) What was the resulting concentration?\nQ4. (About Passage D) What was the humidity percentage?\nQ5. (About Passage E) What direction was the wind from?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Monday\", \"2\": \"Lumi\", \"3\": \"45.1\", \"4\": \"62\", \"5\": \"east\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"E\", \"passage_idx\": 4, \"question\": \"What day were conditions recorded?\", \"answer_key\": \"day\", \"gold\": \"Monday\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"Who logged the shipment?\", \"answer_key\": \"person\", \"gold\": \"Lumi\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the resulting concentration?\", \"answer_key\": \"concentration\", \"gold\": \"45.1\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What was the humidity percentage?\", \"answer_key\": \"humidity\", \"gold\": \"62\", \"phase\": \"shift\"}, {\"question_num\": 5, \"passage\": \"E\", \"passage_idx\": 4, \"question\": \"What direction was the wind from?\", \"answer_key\": \"direction\", \"gold\": \"east\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_frontier_035",
  "task_type": "inhibition_return",
  "difficulty": "Frontier",
  "prompt": "Read the following 5 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe warehouse at the north dock received a shipment on Tuesday. It contained 36 crates of electronics, each weighing 15 kilograms. The shipment was logged by Wren under reference number WH-7120. Construction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The total value of the shipment was $15,400.\n\nA thin rain began to fall just as Tariq reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\n--- Passage B ---\nThe weather station at the hilltop recorded conditions on Thursday. Temperature reached 31.7°C with humidity at 87%. Wind speed was measured at 42 km/h from the east. The clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Barometric pressure stood at 1015 hPa.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ines fully understood. The wooden shelves bowed slightly under the weight.\n\n--- Passage C ---\nExperiment EX-712 was conducted by Freya in lab A-18. The sample was heated to 80°C for 90 minutes. The resulting concentration was 12.4 mg/L. The market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The pH of the solution measured 4.5 at completion.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Priya fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\n--- Passage D ---\nThe weather station at the coastal station recorded conditions on Wednesday. Temperature reached 8.9°C with humidity at 55%. Wind speed was measured at 52 km/h from the northeast. Evening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Barometric pressure stood at 1008 hPa.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\n--- Passage E ---\nExperiment EX-401 was conducted by Nalini in lab B-12. The sample was heated to 65°C for 90 minutes. The resulting concentration was 45.1 mg/L. A street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The pH of the solution measured 6.2 at completion.\n\nAnswer each question:\nQ1. (About Passage C) What temperature was the sample heated to?\nQ2. (About Passage E) What was the experiment ID?\nQ3. (About Passage A) How many crates were in the shipment?\nQ4. (About Passage E) What temperature was the sample heated to?\nQ5. (About Passage C) Who conducted the experiment?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"80\", \"2\": \"EX-401\", \"3\": \"36\", \"4\": \"65\", \"5\": \"Freya\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What temperature was the sample heated to?\", \"answer_key\": \"temp\", \"gold\": \"80\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"E\", \"passage_idx\": 4, \"question\": \"What was the experiment ID?\", \"answer_key\": \"exp_id\", \"gold\": \"EX-401\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"How many crates were in the shipment?\", \"answer_key\": \"quantity\", \"gold\": \"36\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"E\", \"passage_idx\": 4, \"question\": \"What temperature was the sample heated to?\", \"answer_key\": \"temp\", \"gold\": \"65\", \"phase\": \"shift\"}, {\"question_num\": 5, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"Who conducted the experiment?\", \"answer_key\": \"person\", \"gold\": \"Freya\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_frontier_036",
  "task_type": "inhibition_return",
  "difficulty": "Frontier",
  "prompt": "Read the following 5 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nExperiment EX-350 was conducted by Yara in lab B-12. The sample was heated to 120°C for 45 minutes. The resulting concentration was 45.1 mg/L. The workshop on Femi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The pH of the solution measured 5.8 at completion.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Olena fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Gael Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\n--- Passage B ---\nThe warehouse at the north dock received a shipment on Tuesday. It contained 60 crates of chemicals, each weighing 32 kilograms. The shipment was logged by Ravi under reference number WH-4017. Evening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The total value of the shipment was $9,100.\n\nThe workshop on Elio Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Gael reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\n--- Passage C ---\nThe weather station at the valley floor recorded conditions on Friday. Temperature reached 12.4°C with humidity at 62%. Wind speed was measured at 42 km/h from the north. Evening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Barometric pressure stood at 1008 hPa.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ravi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\n--- Passage D ---\nThe weather station at the hilltop recorded conditions on Thursday. Temperature reached 18.5°C with humidity at 62%. Wind speed was measured at 42 km/h from the northeast. Fog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Barometric pressure stood at 1003 hPa.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Willa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\n--- Passage E ---\nThe warehouse at building 7 received a shipment on Monday. It contained 15 crates of ceramics, each weighing 18 kilograms. The shipment was logged by Gael under reference number WH-2293. Fog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The total value of the shipment was $9,100.\n\nAnswer each question:\nQ1. (About Passage E) How much did each crate weigh?\nQ2. (About Passage D) What was the humidity percentage?\nQ3. (About Passage A) What was the final pH?\nQ4. (About Passage C) What was the barometric pressure?\nQ5. (About Passage E) What was the reference number?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"18\", \"2\": \"62\", \"3\": \"5.8\", \"4\": \"1008\", \"5\": \"WH-2293\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"E\", \"passage_idx\": 4, \"question\": \"How much did each crate weigh?\", \"answer_key\": \"weight\", \"gold\": \"18\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What was the humidity percentage?\", \"answer_key\": \"humidity\", \"gold\": \"62\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the final pH?\", \"answer_key\": \"ph\", \"gold\": \"5.8\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the barometric pressure?\", \"answer_key\": \"pressure\", \"gold\": \"1008\", \"phase\": \"shift\"}, {\"question_num\": 5, \"passage\": \"E\", \"passage_idx\": 4, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-2293\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_frontier_037",
  "task_type": "inhibition_return",
  "difficulty": "Frontier",
  "prompt": "Read the following 5 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe warehouse at the central depot received a shipment on Friday. It contained 72 crates of machinery parts, each weighing 32 kilograms. The shipment was logged by Dariush under reference number WH-3506. Evening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The total value of the shipment was $4,200.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\n--- Passage B ---\nThe weather station at the coastal station recorded conditions on Wednesday. Temperature reached 12.4°C with humidity at 78%. Wind speed was measured at 42 km/h from the northeast. A shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Barometric pressure stood at 1021 hPa.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\n--- Passage C ---\nExperiment EX-884 was conducted by Elio in lab D-03. The sample was heated to 65°C for 90 minutes. The resulting concentration was 33.6 mg/L. The old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The pH of the solution measured 5.8 at completion.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Elara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\n--- Passage D ---\nThe warehouse at the east wing received a shipment on Thursday. It contained 15 crates of textiles, each weighing 18 kilograms. The shipment was logged by Magnus under reference number WH-4017. A street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The total value of the shipment was $4,200.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Idris fully understood. The wooden shelves bowed slightly under the weight.\n\n--- Passage E ---\nExperiment EX-350 was conducted by Celine in lab C-05. The sample was heated to 45°C for 15 minutes. The resulting concentration was 33.6 mg/L. The botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The pH of the solution measured 6.2 at completion.\n\nAnswer each question:\nQ1. (About Passage D) Who logged the shipment?\nQ2. (About Passage A) How many crates were in the shipment?\nQ3. (About Passage A) How much did each crate weigh?\nQ4. (About Passage E) What was the final pH?\nQ5. (About Passage D) What was the reference number?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Magnus\", \"2\": \"72\", \"3\": \"32\", \"4\": \"6.2\", \"5\": \"WH-4017\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"Who logged the shipment?\", \"answer_key\": \"person\", \"gold\": \"Magnus\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"How many crates were in the shipment?\", \"answer_key\": \"quantity\", \"gold\": \"72\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"How much did each crate weigh?\", \"answer_key\": \"weight\", \"gold\": \"32\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"E\", \"passage_idx\": 4, \"question\": \"What was the final pH?\", \"answer_key\": \"ph\", \"gold\": \"6.2\", \"phase\": \"shift\"}, {\"question_num\": 5, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-4017\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_frontier_038",
  "task_type": "inhibition_return",
  "difficulty": "Frontier",
  "prompt": "Read the following 5 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe warehouse at building 7 received a shipment on Wednesday. It contained 24 crates of chemicals, each weighing 8 kilograms. The shipment was logged by Elara under reference number WH-8841. The clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The total value of the shipment was $9,100.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Qadir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\n--- Passage B ---\nExperiment EX-712 was conducted by Amara in lab A-18. The sample was heated to 95°C for 30 minutes. The resulting concentration was 28.7 mg/L. Fog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The pH of the solution measured 6.2 at completion.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\n--- Passage C ---\nThe weather station at the hilltop recorded conditions on Tuesday. Temperature reached 23.2°C with humidity at 55%. Wind speed was measured at 52 km/h from the north. The botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Barometric pressure stood at 1015 hPa.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Tala reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\n--- Passage D ---\nExperiment EX-229 was conducted by Uma in lab B-22. The sample was heated to 150°C for 45 minutes. The resulting concentration was 12.4 mg/L. Construction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The pH of the solution measured 6.2 at completion.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\n--- Passage E ---\nThe warehouse at the east wing received a shipment on Tuesday. It contained 60 crates of chemicals, each weighing 18 kilograms. The shipment was logged by Ines under reference number WH-7120. Construction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The total value of the shipment was $9,100.\n\nAnswer each question:\nQ1. (About Passage A) What was the reference number?\nQ2. (About Passage D) What was the final pH?\nQ3. (About Passage E) How many crates were in the shipment?\nQ4. (About Passage C) What was the humidity percentage?\nQ5. (About Passage A) How many crates were in the shipment?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"WH-8841\", \"2\": \"6.2\", \"3\": \"60\", \"4\": \"55\", \"5\": \"24\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What was the reference number?\", \"answer_key\": \"ref\", \"gold\": \"WH-8841\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What was the final pH?\", \"answer_key\": \"ph\", \"gold\": \"6.2\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"E\", \"passage_idx\": 4, \"question\": \"How many crates were in the shipment?\", \"answer_key\": \"quantity\", \"gold\": \"60\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"C\", \"passage_idx\": 2, \"question\": \"What was the humidity percentage?\", \"answer_key\": \"humidity\", \"gold\": \"55\", \"phase\": \"shift\"}, {\"question_num\": 5, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"How many crates were in the shipment?\", \"answer_key\": \"quantity\", \"gold\": \"24\", \"phase\": \"return\"}]}"
 },
 {
  "task_id": "ior_frontier_039",
  "task_type": "inhibition_return",
  "difficulty": "Frontier",
  "prompt": "Read the following 5 passages carefully, then answer each question.\nEach question specifies which passage it refers to.\n\n--- Passage A ---\nThe weather station at the hilltop recorded conditions on Thursday. Temperature reached 31.7°C with humidity at 45%. Wind speed was measured at 15 km/h from the west. A thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Barometric pressure stood at 997 hPa.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Hana Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\n--- Passage B ---\nExperiment EX-401 was conducted by Freya in lab A-18. The sample was heated to 65°C for 45 minutes. The resulting concentration was 8.9 mg/L. The botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The pH of the solution measured 7.1 at completion.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Oulu insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\n--- Passage C ---\nThe warehouse at the east wing received a shipment on Wednesday. It contained 36 crates of textiles, each weighing 32 kilograms. The shipment was logged by Bram under reference number WH-3506. A shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The total value of the shipment was $12,300.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\n--- Passage D ---\nThe warehouse at the north dock received a shipment on Tuesday. It contained 15 crates of electronics, each weighing 25 kilograms. The shipment was logged by Freya under reference number WH-7120. The market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The total value of the shipment was $7,850.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Vesna Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Bashir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\n--- Passage E ---\nThe warehouse at the east wing received a shipment on Thursday. It contained 15 crates of chemicals, each weighing 8 kilograms. The shipment was logged by Sigrid under reference number WH-7120. The workshop on Tariq Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The total value of the shipment was $4,200.\n\nAnswer each question:\nQ1. (About Passage D) Who logged the shipment?\nQ2. (About Passage B) What was the final pH?\nQ3. (About Passage B) Who conducted the experiment?\nQ4. (About Passage A) What direction was the wind from?\nQ5. (About Passage D) What day did the shipment arrive?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Freya\", \"2\": \"7.1\", \"3\": \"Freya\", \"4\": \"west\", \"5\": \"Tuesday\"}, \"questions\": [{\"question_num\": 1, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"Who logged the shipment?\", \"answer_key\": \"person\", \"gold\": \"Freya\", \"phase\": \"initial\"}, {\"question_num\": 2, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"What was the final pH?\", \"answer_key\": \"ph\", \"gold\": \"7.1\", \"phase\": \"shift\"}, {\"question_num\": 3, \"passage\": \"B\", \"passage_idx\": 1, \"question\": \"Who conducted the experiment?\", \"answer_key\": \"person\", \"gold\": \"Freya\", \"phase\": \"shift\"}, {\"question_num\": 4, \"passage\": \"A\", \"passage_idx\": 0, \"question\": \"What direction was the wind from?\", \"answer_key\": \"direction\", \"gold\": \"west\", \"phase\": \"shift\"}, {\"question_num\": 5, \"passage\": \"D\", \"passage_idx\": 3, \"question\": \"What day did the shipment arrive?\", \"answer_key\": \"day\", \"gold\": \"Tuesday\", \"phase\": \"return\"}]}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['shifting', 'inhibition_return']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "shifting": cogattention_shifting,
    "inhibition_return": cogattention_inhibition_return,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Attention Shifting")
